In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('../../KM.csv')
df

,ECNumber,Organism,Smiles,substrate,Sequence,Type,Source,Value,Test,UniprotID,pH,Temperature,log10_KM,KEGG ID
0,6.2.1.26,Bacillus subtilis,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,coa,MLTEQPNWLMQRAQLTPERIALIYEDQTVTFAELFAASKRMAEQLA...,mutant,custom,0.1700,0,P23971,7.5,NaN,-0.769551,C00010
1,1.1.1.22,Burkholderia cepacia,O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[C@H...,udp-alpha-d-glucose,MNLTIIGSGYVGLVTGACLADIGHDVFCLDVDQAKIDILNNGGVPI...,wild,custom,0.2300,0,C9E261,8.7,NaN,-0.638272,C00029
2,4.6.1.2,Homo sapiens,Nc1nc2c(ncn2[C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,gtp,MFCTKLKDLKITGECPFSLLAPGQVPNESSEEAAGSSESCKATVPI...,wild,custom,0.0650,0,Q02108,7.5,NaN,-1.187087,C00044
3,4.4.1.13,Escherichia coli,N[C@@H](CCSC[C@H](N)C(=O)O)C(=O)O,l-cystathionine,MADKKLDTQLVNAGRSKKYTLGAVNSVIQRASSLVFDSVEAKKHAT...,mutant,custom,0.1010,0,P06721,8.5,NaN,-0.995679,C02291
4,1.5.1.3,Plasmodium falciparum,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,nadph,MMEQVCDVFDIYAICACCKVESKNEGKKNEVFNNQTRRGLGNKGVL...,mutant,custom,0.0105,0,P13922,5.0,NaN,-1.978811,C00005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20774,2.1.1.20,Rattus norvegicus,NCC(=O)O,gly,MVDSVYRTRSLGVAAEGIPDQYADGEAARVWQLYIGDTRSRTAEYK...,mutant,custom,2.8000,1,P13255,7.2,NaN,0.447158,C00037
20775,5.1.1.1,Escherichia coli,C[C@@H](N)C(=O)O,d-alanine,MQAATVVINRRALRHNLQRLRELAPASKMVAVVKANAYGHGLLETA...,mutant,custom,0.6040,1,P0A6B4,NaN,NaN,-0.218963,C00133
20776,2.5.1.54,Pyrococcus furiosus,C=C(OP(=O)(O)O)C(=O)O,phosphoenolpyruvate,MKYSKEYKEKTVVKINDVKFGEGFTIIAGPCSIESRDQIMKVAEFL...,mutant,custom,0.0660,1,Q8U0A9,7.5,NaN,-1.180456,C00074
20777,6.2.1.71,Escherichia coli,O=C(O)c1ccccc1O,salicylate,MSIPFTRWPEEFARRYREKGYWQDLPLTDILTRHAASDSIAVIDGE...,mutant,custom,0.0089,1,P10378,8.0,NaN,-2.050610,C00805


In [2]:
"""
Runs embedding/inference for an input fasta file:

Code adapted from original (written by smsaladi) found at https://github.com/smsaladi/UniRep
"""


import numpy as np
import pandas as pd

import tensorflow as tf

from Bio import SeqIO
from Bio import AlignIO

import unirep0
import data_utils

from pkg_resources import resource_filename


class BatchBabbler1900(unirep0.babbler1900):
    '''
    Subclass babbler to replace the get_rep method.
    '''

    def __init__(self, batch_size=32, model_path=resource_filename(__name__, "1900_weights")):
        super().__init__(batch_size=batch_size, model_path=model_path)

    def get_rep(self, seqs, sess):
        """
        Monkey-patch get_rep to accept a tensorflow session (instead of initializing one each time)
        """
        if isinstance(seqs, str):
            seqs = pd.Series([seqs])

        coded_seqs = [aa_seq_to_int(s) for s in seqs]
        n_seqs = len(coded_seqs)

        if n_seqs == self._batch_size:
            zero_batch = self._zero_state
        else:
            zero = self._zero_state[0]
            zero_batch = [zero[:n_seqs,:], zero[:n_seqs, :]]

        final_state_, hs = sess.run(
                [self._final_state, self._output], feed_dict={
                    self._batch_size_placeholder: n_seqs,
                    self._minibatch_x_placeholder: coded_seqs,
                    self._initial_state_placeholder: zero_batch
                })

        final_cell, final_hidden = final_state_
        avg_hidden = np.mean(hs, axis=1)

        df = seqs.to_frame()
        df['avg_hs'] = np_to_list(avg_hidden)[:n_seqs]
        df['final_hs'] = np_to_list(final_hidden)[:n_seqs]
        df['final_cell'] = np_to_list(final_cell)[:n_seqs]

        return df


class BatchInference(object):
    '''
    A class for getting UniRep50 embeddings in batches.
    The main idea is to group sequences of the same length.
    This speeds up things quite a lot.
    '''
    def __init__(self, batch_size, model_path=resource_filename(__name__, "1900_weights")):
        self.batch_size = batch_size
        self.model_path = model_path

        # initialize the babbler
        self.bab = BatchBabbler1900(batch_size=self.batch_size, model_path=self.model_path)

    def run_inference(self, filepath):
        '''
        '''
        # read sequences into a Pandas series with sequences and identifiers
        seqs = series_from_seqio(filepath, 'fasta')
        seqs = seqs.str.rstrip('*')
        df_seqs = seqs.to_frame()

        # save starting index for re-sorting frame at the end
        old_index = df_seqs.index # save old index (sequence header values) for later use in re-sorting

        # sort by length
        df_seqs['len'] = df_seqs['seq'].str.len()
        df_seqs.sort_values('len', inplace=True)
        index = df_seqs.index # save index (sequence header values) for later use
        df_seqs.reset_index(drop=True, inplace=True)
        df_seqs['grp'] = df_seqs.groupby('len')['len'].transform(lambda x: np.arange(np.size(x))) // self.batch_size

        # set up tf session, then run inference
        with tf.compat.v1.Session() as sess:
            unirep0.initialize_uninitialized(sess)
            df_calc = df_seqs.groupby(['grp', 'len'], as_index=False, sort=False).apply(lambda d: self.bab.get_rep(seqs=d['seq'], sess=sess))

        # expand out the lists so each value gets its own cell
        av = df_calc['avg_hs'].apply(pd.Series)
        av.columns = ['av_{0}'.format(i+1) for i in range(1900)]

        fh = df_calc['final_hs'].apply(pd.Series)
        fh.columns = ['fh_{0}'.format(i+1) for i in range(1900)]

        fc = df_calc['final_cell'].apply(pd.Series)
        fc.columns = ['fc_{0}'.format(i+1) for i in range(1900)]

        out_df = pd.concat([av, fh, fc], axis=1)
        out_df.index = index

        return out_df.reindex(old_index)


def series_from_seqio(fn, format, **kwargs):
    if format in SeqIO._FormatToIterator.keys():
        reader = SeqIO.parse
    elif format in AlignIO._FormatToIterator.keys():
        reader = AlignIO.read
    else:
        raise ValueError("format {} not recognized by either SeqIO or AlignIO".format(format))

    if isinstance(fn, str) and 'gz' in fn:
        with gzip.open(fn, "rt") as fh:
            seqs = reader(fh, format, *kwargs)
    else:
        seqs = reader(fn, format, *kwargs)

    seqs = [(r.description, str(r.seq).upper()) for r in seqs]
    seqs = list(zip(*seqs))
    seqs = pd.Series(seqs[1], index=seqs[0], name="seq")

    return seqs


def np_to_list(arr):
    return [arr[i] for i in np.ndindex(arr.shape[:-1])]


def aa_seq_to_int(s):
    """
    Monkey patch to return unknown if not in alphabet
    """
    s = s.strip()
    s_int = [24] + [data_utils.aa_to_int.get(a, data_utils.aa_to_int['X']) for a in s] + [25]
    return s_int[:-1]


In [3]:
import sys
sys.path.append('../')

In [13]:
from functions_for_unirep_calculations import compute_unirep_representations,add_Unirep_vector

In [18]:
datasets_dir = '../../../datasets'
import os
ofile = open(os.path.join(datasets_dir, "enzyme_data", "all_sequences_sabio.fasta"), "w")
for ind in df.index:
    seq = df["Sequence"][ind]
    if not pd.isnull(seq):
        seq_end = seq.find("#")
        ofile.write(">" + str(ind) + "\n" +seq[:seq_end] + "\n")
ofile.close()
X_unirep = compute_unirep_representations(fasta_file = os.path.join(datasets_dir, "enzyme_data", "all_sequences_sabio.fasta"),
                                          rep_file = os.path.join(datasets_dir, "enzyme_data", "unirep_representations_sabio.tsv"))
Unirep_df = pd.read_csv(os.path.join(datasets_dir, "enzyme_data", "unirep_representations_sabio.tsv"), sep = "\t")
df = add_Unirep_vector(df, Unirep_df)
df.head()



Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API

The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.

Instructions for updating:
Please use `layer.__call__` method instead.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributions`.
Instructions for updating:
The TensorFlow Distributions library has moved to TensorFlow Probability (https://github.com/tensorflow/probability). You should update all references to use `tfp.distributions` instead of `tf.distributi

2024-12-05 19:46:40.201048: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcuda.so.1
2024-12-05 19:46:40.296412: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1618] Found device 0 with properties: 
name: NVIDIA A800 80GB PCIe major: 8 minor: 0 memoryClockRate(GHz): 1.41
pciBusID: 0000:39:00.0
2024-12-05 19:46:40.298048: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1618] Found device 1 with properties: 
name: NVIDIA A800 80GB PCIe major: 8 minor: 0 memoryClockRate(GHz): 1.41
pciBusID: 0000:9c:00.0
2024-12-05 19:46:40.299676: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1618] Found device 2 with properties: 
name: NVIDIA A800 80GB PCIe major: 8 minor: 0 memoryClockRate(GHz): 1.41
pciBusID: 0000:9d:00.0
2024-12-05 19:46:40.300859: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1618] Found device 3 with properties: 
name: NVIDIA A800 80GB PCIe major: 8 minor: 0 memoryClockRate(GHz): 1.41
pciBusID: 0000:a0:00.0
202

2024-12-05 19:46:41.538005: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1159] Device interconnect StreamExecutor with strength 1 edge matrix:
2024-12-05 19:46:41.538045: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1165]      
../functions_for_unirep_calculations.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Unirep"][ind] = X_Unirep[i,1:]
/home/wuke/anaconda3/envs/km_pre/lib/python3.7/site-packages/pandas/core/indexing.py:1732: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_block(indexer, value, name)


,ECNumber,Organism,Smiles,substrate,Sequence,Type,Source,Value,Test,UniprotID,pH,Temperature,log10_KM,KEGG ID,Unirep
0,6.2.1.26,Bacillus subtilis,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,coa,MLTEQPNWLMQRAQLTPERIALIYEDQTVTFAELFAASKRMAEQLA...,mutant,custom,0.1700,0,P23971,7.5,NaN,-0.769551,C00010,"[0.018741472, 0.06163029, 0.059035555, -0.0252..."
1,1.1.1.22,Burkholderia cepacia,O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[C@H...,udp-alpha-d-glucose,MNLTIIGSGYVGLVTGACLADIGHDVFCLDVDQAKIDILNNGGVPI...,wild,custom,0.2300,0,C9E261,8.7,NaN,-0.638272,C00029,"[0.03434499, 0.042849235, 0.08976609, -0.04268..."
2,4.6.1.2,Homo sapiens,Nc1nc2c(ncn2[C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,gtp,MFCTKLKDLKITGECPFSLLAPGQVPNESSEEAAGSSESCKATVPI...,wild,custom,0.0650,0,Q02108,7.5,NaN,-1.187087,C00044,"[0.026338853, -0.010613492, 0.06317112, -0.020..."
3,4.4.1.13,Escherichia coli,N[C@@H](CCSC[C@H](N)C(=O)O)C(=O)O,l-cystathionine,MADKKLDTQLVNAGRSKKYTLGAVNSVIQRASSLVFDSVEAKKHAT...,mutant,custom,0.1010,0,P06721,8.5,NaN,-0.995679,C02291,"[0.017732581, 0.16894118, 0.080610655, -0.0636..."
4,1.5.1.3,Plasmodium falciparum,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,nadph,MMEQVCDVFDIYAICACCKVESKNEGKKNEVFNNQTRRGLGNKGVL...,mutant,custom,0.0105,0,P13922,5.0,NaN,-1.978811,C00005,"[0.0030080562, -0.025697082, 0.06640412, -0.01..."


In [19]:
df.to_csv("KM_Unirep.csv", index = False)